# AP 155 Lab Exercise
---
## Exercise 5: Matrices

_Instructions_: 
- **Report figure:** No report figure. Points in the report figure will be allocated to the report discussion 
- **Report discussion:**
  - What were the basis linear equations used to set up the matrix in each question of Problem 2?
  - Once you set up the matrices, what function did you use to solve the matrix equation (Could have different answers)? Check the docstring/documentation/other references and summarize the mechanism behind the function 
- **Code**: Complete the code and ensure it is functional, error-free, readable, and efficient (where needed). Include concise Markdown documentation highlighting the critical steps of your algorithm for each problem.

### Student Information
- _Full Name (Last Name, First Name)_: de Vera, Clarence Joseph
- _Student No._: 2024-11757
- _Section_: THR-TX-3

### Grading Information (c/o Lab Instructor)
- [Rubrics description link](https://drive.google.com/file/d/1BMSlPot2Mc7XLu0eo4S8I8gLBsIadbCL/view?usp=sharing) (Note: percentages may still be tweaked)

| Criteria | Score | Subtotal |
| --- | --- | --- |
| Report figure | XX | 20 |
| Report discussion | XX | 20|
| Code readability | XX | 20 |
| Code efficiency | XX | 20 | 
| Code appropriateness  | XX | 20 | 
| **TOTAL** | XXX | 100 |

_Date and Time Scored (MM/DD/YYYY HH:MM AM/PM):_:_

---
## Section 1: Report

<!-- ### Report figure

<img src="figures/cat.jpg" alt="Alt text" width=45%> <img src="figures/cat.jpg" alt="Alt text" width=45%>


EDIT ME. See instruction above for what to include. Insert caption here, a short description what the figure illustrates. The syntax in markdown for putting a figure is `![Description](local_file_name.png)` or `<img src="/path/to_image.jpg" alt="Alt text" width=45%>`. Make sure the figure has complete elements.
-->

### Report discussion

**1.**
* x-axis (horizontal): $-T_1 \cos(\alpha) + T_2 \cos(\beta) = 0$
* y-axis (vertical): $T_1 \sin(\alpha) + T_2 \sin(\beta) = Mg$

These equations translate into the coefficient matrix `A` and constant matrix `B` in the code.

**2.**
* Function Used: `np.linalg.solve(A, B)`

### Other comments
- EDIT ME
- A self reflection can be placed here.
- This portion is not graded.
- You may insert further questions here.


---
## Section 2: Code

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

### Problem 1: (Circuit Analysis)


Consider a long chain of resistors wired up like this:

<img src="reschain.png" alt="Alt text" width=95%>

All the resistors have the same resistance~$R$.  The power rail at the top is at voltage~$V_+=5$V.  The problem is to find the voltages $V_1...V_N$ at the internal points in the circuit.

1. **Using Ohm's law and the Kirchhoff current law**, which says that the total net current flow out of (or into) any junction in a circuit must be zero, show that the voltages $V_1\ldots V_N$ satisfy the equations

$$3V_1 - V_2 - V_3 = V_+ $$
$$-V_1 + 4V_2 - V_3 - V_4 = V_+ $$
      
$$-V_{i-2} - V_{i-1} + 4V_i - V_{i+1} - V_{i+2} = 0 $$

$$-V_{N-3} - V_{N-2} + 4V_{N-1} - V_N = V_- = 0 $$
$$-V_{N-2} - V_{N-1} + 3V_N = V_- = 0 $$

Note: The assignment of $V_-$ is to make the equations symmetric

2. Express these equations in vector form~$A\vec{v} = \vec{w}$ and find the values of the matrix~$A$ and the vector~$\vec{w}$.

3. Write a program to solve for the values of the~$V_i$ when there are $N=6$ internal junctions with unknown voltages.  (Hint: All the values of $V_i$ should lie between zero and $5$V.  If they don't, something is wrong.)

4. Now repeat your calculation for the case where there are $N=10\,000$ internal junctions.  This part is not possible using standard tools like the *solve* function.  You need to make use of the fact that the matrix~$\vec{A}$ is banded.  

1.
 Kirchhoff's current law: the total electrical current entering a circuit junction equals the total current leaving that same junction.
 
 Ohm's law: (I = V/R)

   - Since all the resistors have the same R, the denominator can be ignored. Therefore, it's clear that the voltage in a certain junction is equal to the voltage in the neighboring junctions connected to it.
   - Example: junction V_1 connects to V_2, V_3, and V_+. Summing currents: (V_1 - V_+) + (V_1 - V_2) + (V_1 - V_3) = 0 => 3V_1 - V_2 - V_3 = V_+

In [2]:
import numpy as np

def build_full_matrix(N):
    A = np.zeros((N, N))
    
    # diagonal
    np.fill_diagonal(A, 4)
    A[0, 0] = 3
    A[-1, -1] = 3
    
    # first and second off-diagonals (-1)
    for i in range(N):
        if i + 1 < N:
            A[i, i + 1] = -1
            A[i + 1, i] = -1
        if i + 2 < N:
            A[i, i + 2] = -1
            A[i + 2, i] = -1
            
    return A

N = 6
A = build_full_matrix(N)

# w vector
V_plus = 5.0
w = np.zeros(N)
w[0] = V_plus
w[1] = V_plus

print("Vector Form (6x6):")
print(A)

Vector Form (6x6):
[[ 3. -1. -1.  0.  0.  0.]
 [-1.  4. -1. -1.  0.  0.]
 [-1. -1.  4. -1. -1.  0.]
 [ 0. -1. -1.  4. -1. -1.]
 [ 0.  0. -1. -1.  4. -1.]
 [ 0.  0.  0. -1. -1.  3.]]


In [3]:
v = np.linalg.solve(A, w)
print("Value of voltages (V_1 to V_6):")
print(v)

Value of voltages (V_1 to V_6):
[3.7254902  3.43137255 2.74509804 2.25490196 1.56862745 1.2745098 ]


In [4]:
from scipy.linalg import solve_banded
N_band = 10000
w_band = np.zeros(N_band)
w_band[0] = V_plus
w_band[1] = V_plus

ab = np.zeros((5, N_band))

ab[0, 2:] = -1  # second off-diagonal (forward)
ab[1, 1:] = -1  # first off-diagonal (forward)
ab[2, :] = 4    # diagonal (4)
ab[2, 0] = 3    # first diagonal (3)
ab[2, -1] = 3   # last diagonal (3)
ab[3, :-1] = -1 # second off-diagonal (backward)
ab[4, :-2] = -1 # second off-diagonal (backward)

V_10000 = solve_banded((2, 2), ab, w_band)

print("First 5 voltages for N = 10000:")
print(np.round(V_10000[:5], 3))
print("Last 5 voltages for N = 10000:")
print(np.round(V_10000[-5:], 3))

First 5 voltages for N = 10000:
[4.999 4.999 4.998 4.998 4.997]
Last 5 voltages for N = 10000:
[0.003 0.002 0.002 0.001 0.001]


#### Problem 1 code summary

This script applies Ohm's law and the Kirchhoff current law to determine the voltages V_1 ... V_N at the internal junctions of a resistor chain. The system of equations is expressed in vector form Av=w. To efficiently calculate the case scenario where there are N = 10,000 junctions, the script makes use of the fact that the matrix A is banded.   

---

### Problem 2 (Statics)


Solve the following matrix problems. Set-up the linear equations that describe the problem then solve them 

Part 1: Suppose you have a hanging mass M supported by two ropes (angled by $\alpha$ and $\beta$ with respect to the ceiling). Make a function that calculates the tension on each of the ropes. Put your function in a python file and call it here in the notebook. Check that the answer by computer matches your own computations.


Part 2: 
Masses $m_1, m_2, m_3$, and $m_4$ lie on a $2.00$ [m] beam of negligible mass. They are located $0.20, 0.70, 1.10$, and $1.40$ [m] away from the left end of the beam. Determine the masses $m_1, m_2, m_3$, and $m_4$, given the following constraints: 

- The total mass is $8.00$ [kg]
- If a pivot is placed halfway, the beam will balance if a $1.05$ [kg] mass is placed on the right-end of the beam
- If a pivot is placed $1.20$ [m] away from the right end, then the beam would balance if a $550$ [g] mass is placed on the left end of the beam.
- If the system is split midway, the total mass on the left is $1.00$ [kg] heavier than the total mass on the right

In [5]:
from samplePackage import comptens
import numpy as np

mass = 10.0  # kg
alpha = 30.0 # degrees
beta = 60.0  # degrees
g = 9.81     # m/s^2

T1_comp, T2_comp = comptens(mass, alpha, beta, g)

T1_clark = (mass * g) / 2
T2_clark = T1_clark * np.sqrt(3)

print(f"Computer T1: {T1_comp:.3f} N  |  Clark T1: {T1_clark:.3f} N")
print(f"Computer T2: {T2_comp:.3f} N  |  Clark T2: {T2_clark:.3f} N")

assert np.isclose(T1_comp, T1_clark), "T1 values do not match!"
assert np.isclose(T2_comp, T2_clark), "T2 values do not match!"
print("Computer outputs matches own calculations!")

Computer T1: 49.050 N  |  Clark T1: 49.050 N
Computer T2: 84.957 N  |  Clark T2: 84.957 N
Computer outputs matches own calculations!


#### Problem 2 code summary

This script solves the matrix problem for a hanging mass M supported by two ropes angled by $\alpha$ and $\beta$ by setting up the linear equations that describe the system. The function was created in a separate Python file and called in the notebook. The notebook is then used to check that the answer by the computer matches my own manual computations.   

---

## Section 3 (Notes)

### Some codes for solving matrix problems

* solving matrix equations ($\bf{A}\vec{x} = \vec{b}$) -> `np.linalg.solve`
* LU-decomposition ($\bf{A} = \bf{L}\bf{U}$) -> `scipy.linalg.lu`
* matrix inversion ($\bf{A}^{-1}$) -> `np.linalg.inv`
* QR-decomposition ($\bf{A} = \bf{Q}\bf{R}$) -> `scipy.linalg.qr`
* eigenvalue problem ($\bf{A}\vec{x} = \lambda \vec{x}$) -> `np.linalg.eig`

Solving matrix problems by themselves isn't difficult. There is an abundance of linear algebra libraries that are ready to use: from the battle-tested ones (the classic [BLAS](https://www.netlib.org/blas/) and [LAPACK](https://www.netlib.org/lapack/)), HPC tools for large problems ([ScaLAPACK](https://netlib.org/scalapack/)), to hardware-accelerated ones often with GPUs ([cuBLAS](https://docs.nvidia.com/cuda/cublas/index.html) and [cuSolver](https://docs.nvidia.com/cuda/cusolver/index.html) for NVIDIA). 

The hardest part is actually composing the matrix. How do you transform a physical problem into a linear algebra problem?

* Circuit: https://personal.math.vt.edu/embree/cmda3606/chapter2.pdf
* Embree [main notes](https://personal.math.vt.edu/embree/cmda3606notes.pdf) + [lab manual](https://personal.math.vt.edu/embree/labman.pdf)

#### Sample Usage (Testing different codes)

In [6]:
A = np.array([[1,0,2],[-2,-1,3],[0,3,5]])
B = np.array([[-2],[1],[-3]])


In [7]:
sigma_z = [[1,0],[0,-1]]
sigma_y = [[0,-1j],[1j,0]]
sigma_x = [[0,1],[1,0]]

In [8]:
ans = np.linalg.solve(sigma_x, sigma_y)
ans = np.linalg.inv(A) @ B


In [9]:
from scipy.linalg import *
p, l, u = lu(A)
p, l, u

(array([[0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.]]),
 array([[ 1.        ,  0.        ,  0.        ],
        [-0.        ,  1.        ,  0.        ],
        [-0.5       , -0.16666667,  1.        ]]),
 array([[-2.        , -1.        ,  3.        ],
        [ 0.        ,  3.        ,  5.        ],
        [ 0.        ,  0.        ,  4.33333333]]))

### Importing functions from a python file

In [10]:
import samplePackage as sP

### np.roll

In [11]:
## Useful tips. Not required to solve the problems below!

# Suppose you have a Numpy array
x = np.zeros((4,4))
x[1,2] = 5
for row in x:
    print(row)

#rint(x[0,1:3])


# I can move all values to the left/right using roll
x_roll = np.roll(x, 1) # change 1 to -1, what happens? 
#x_roll2 = np.roll(x, -1)
print(x_roll)

x_roll2 = np.roll(x, 2)
print(x_roll2)

[0. 0. 0. 0.]
[0. 0. 5. 0.]
[0. 0. 0. 0.]
[0. 0. 0. 0.]
[[0. 0. 0. 0.]
 [0. 0. 0. 5.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [5. 0. 0. 0.]
 [0. 0. 0. 0.]]
